# Creating Silver Tables

The purpose of this notebook is to create three silver layer tables for the Retail Sales ETL Pipeline.

The following steps were completed:
1. Loaded the three bronze layer tables from the previous step into DataFrames.
2. Cast each DataFrame's fields to proper data types.
3. Filtered out NULL values and Invalid records.
4. Standardized values within each row.
5. Added new analytic columns.
6. Wrote the cleaned data to three Silver layer Delta tables.

### Loading the bronze tables as DataFrames

In [0]:
customers_df = spark.read.table("workspace.retail_schema.customer_bronze")
customers_df.head(5)

[Row(Customer_ID='NM/18520', Customer_Name='Neoma Murray', Segment='Consumer', Country='United States', City='Riverside', State='California', Postal_Code=92503, Region='West', Age=33),
 Row(Customer_ID='ES/14020', Customer_Name='Erica Smith', Segment='Consumer', Country='United States', City='San Francisco', State='California', Postal_Code=94110, Region='West', Age=32),
 Row(Customer_ID='BM/11650', Customer_Name='Brian Moss', Segment='Corporate', Country='United States', City='New York City', State='New York', Postal_Code=10035, Region='East', Age=43),
 Row(Customer_ID='JG/15310', Customer_Name='Jason Gross', Segment='Corporate', Country='United States', City='Providence', State='Rhode Island', Postal_Code=2908, Region='East', Age=20),
 Row(Customer_ID='CC/12430', Customer_Name='Chuck Clark', Segment='Home Office', Country='United States', City='Columbus', State='Indiana', Postal_Code=47201, Region='Central', Age=39)]

In [0]:
customers_df.count()

793

In [0]:
products_df = spark.read.table("workspace.retail_schema.products_bronze")
products_df.head(5)

[Row(Product_ID='OFF-AP-10004785', Category='Office Supplies', Sub_Category='Appliances', Product_Name='Holmes Replacement Filter for HEPA Air Cleaner, Medium Room'),
 Row(Product_ID='FUR-FU-10002107', Category='Furniture', Sub_Category='Furnishings', Product_Name='Eldon Pizzaz Desk Accessories'),
 Row(Product_ID='OFF-AR-10004930', Category='Office Supplies', Sub_Category='Art', Product_Name='Turquoise Lead Holder with Pocket Clip'),
 Row(Product_ID='FUR-TA-10004152', Category='Furniture', Sub_Category='Tables', Product_Name='"Barricks 18"" x 48"" Non-Folding Utility Table with Bottom Storage Shelf"'),
 Row(Product_ID='OFF-LA-10004544', Category='Office Supplies', Sub_Category='Labels', Product_Name='Avery 505')]

In [0]:
store_df = spark.read.table("workspace.retail_schema.store_bronze")
store_df.head(5)

[Row(Row_ID=1021, Order_ID='CA-2016-124450', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='GT/14710', Product_ID=None, Sales=341100.0, Discount=0.01, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)),
 Row(Row_ID=2737, Order_ID='CA-2016-115798', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='KL/16645', Product_ID=None, Sales=634200.0, Discount=0.03, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)),
 Row(Row_ID=2936, Order_ID='US-2017-169040', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='GT/14710', Product_ID=None, Sales=538350.0, Discount=0.03, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)),
 Row(Row_ID=3139, Order_ID='CA-2018-164168', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='LS/16975', Product_ID=None, Sales=671760.0, Discount=0.02, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)),
 Row

### Cleaning the bronze layer customers table to create the silver layer customers table

In [0]:
from pyspark.sql.functions import when, col, current_timestamp, upper

# Editing the schema to contain the proper data types
customers_clean = customers_df.select(
  col("Customer_ID").cast("string"),
  col("Customer_Name").cast("string"),
  col("Segment").cast("string"),
  col("Country").cast("string"),
  col("City").cast("string"),
  col("State").cast("string"),
  col("Postal_Code").cast("int"),
  col("Region").cast("string"),
  col("Age").cast("int"),
  col("Customer_Ingested_At").cast("timestamp")
)

# Handling Null Values and Invalid records
customers_clean = customers_clean.filter(
  (col("Customer_ID").isNotNull()) &
  (col("Customer_Name").isNotNull()) &
  (col("Age")>0)
)

# Removing duplicates
customers_clean = customers_clean.drop_duplicates(["Customer_ID", "Customer_Name"])

# Standardizing Data
customers_clean = customers_clean.withColumn("Country", upper(col("Country")))
customers_clean = customers_clean.withColumn("City", upper(col("City")))
customers_clean = customers_clean.withColumn("State", upper(col("State")))

# Adding New Columns
customers_clean = customers_clean.withColumn("Age_Group", when(col("Age")<18, "Under 18").when((col("Age")>=18) & (col("Age")<=24), "18-24").when((col("Age")>=25) & (col("Age")<=34), "25-34").when((col("Age")>=35) & (col("Age")<=44), "35-44").when((col("Age")>=45) & (col("Age")<=54), "45-54").when((col("Age")>=55) & (col("Age")<=64), "55-64").when(col("Age")>=65, "65+").otherwise("Unknown"))

# Writing to the Silver Table
customers_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.customers_silver")

### Cleaning the bronze layer products table to create a silver layer products Table

In [0]:
# Editing the schema to contain the proper data types
products_clean = products_df.select(
    col("Product_ID").cast("string"),
    col("Category").cast("string"),
    col("Sub_Category").cast("string"),
    col("Product_Name").cast("string"),
    col("Product_Ingested_At").cast("timestamp")
)

# Filtering out Null and Invalid records
products_clean = products_clean.filter(
    col("Product_ID").isNotNull() &
    col("Category").isNotNull() &
    col("Sub_Category").isNotNull() &
    col("Product_Name").isNotNull()
)

# Removing duplicates
products_clean = products_clean.drop_duplicates(["Product_ID", "Product_Name"])

# Standardizing Data
products_clean = products_clean.withColumn("Category", upper(col("Category")))
products_clean = products_clean.withColumn("Sub_Category", upper(col("Sub_Category")))
products_clean = products_clean.withColumn("Product_Name", upper(col("Product_Name")))

# Writing to Silver Table
products_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.products_silver")

### Cleaning the bronze layer store table to create a silver layer store table

In [0]:
from pyspark.sql.functions import datediff
from pyspark.sql.functions import when, col, current_timestamp, upper

# Editing the schema to contain the proper data types
store_clean = store_df.select(
  col("Row_ID").cast("int"),
  col("Order_ID").cast("string"),
  col("Order_Date").cast("date"),
  col("Ship_Date").cast("date"),
  col("Ship_Mode").cast("string"),
  col("Customer_ID").cast("string"),
  col("Product_ID").cast("string"),
  col("Sales").cast("double"),
  col("Discount").cast("double"),
  col("Store_Ingested_At").cast("timestamp")
)

# Filtering out Null and Invalid records
store_clean = store_clean.filter(
  (col("Row_ID").isNotNull()) &
  (col("Order_ID").isNotNull()) &
  (col("Order_Date").isNotNull()) &
  (col("Ship_Date").isNotNull()) &
  (col("Ship_Mode").isNotNull()) &
  (col("Customer_ID").isNotNull()) &
  (col("Product_ID").isNotNull()) &
  (col("Sales")>0)
)

# Removing duplicates
store_clean = store_clean.drop_duplicates(["Row_ID"])

# Standardizing Data
store_clean = store_clean.withColumn("Ship_Mode", upper(col("Ship_Mode")))

# Adding New Columns
store_clean = store_clean.withColumn("Shipping_Delivery_Days", datediff("Ship_Date", "Order_Date"))
store_clean = store_clean.withColumn("Delivery_Speed_Category", when(col("Shipping_Delivery_Days")<=2, "Fast").when(col("Shipping_Delivery_Days")<=5, "Normal").when(col("Shipping_Delivery_Days")>5, "Slow"))

# Writing to Silver Table
store_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.store_silver")

[Row(Row_ID=1021, Order_ID='CA-2016-124450', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='GT/14710', Product_ID=None, Sales=341100.0, Discount=0.01, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)), Row(Row_ID=2737, Order_ID='CA-2016-115798', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='KL/16645', Product_ID=None, Sales=634200.0, Discount=0.03, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)), Row(Row_ID=2936, Order_ID='US-2017-169040', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='GT/14710', Product_ID=None, Sales=538350.0, Discount=0.03, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)), Row(Row_ID=3139, Order_ID='CA-2018-164168', Order_Date=None, Ship_Date=None, Ship_Mode='Standard Class', Customer_ID='LS/16975', Product_ID=None, Sales=671760.0, Discount=0.02, Store_Ingested_At=datetime.datetime(2025, 7, 29, 19, 51, 47, 831023)), Row(Row